# Leadership and Management Book Recommendation System

## Notebook 05 — SQL Database and EER Design

### Purpose

This notebook transforms the integrated book datasets into relational tables suitable for storage and analysis in MySQL.

The database design separates book-level information from authors, categories, editions, metrics, and source provenance. This reduces unnecessary duplication while preserving the relationships discovered during data integration.

### Objectives

1. Load and validate the final integrated datasets.
2. Review the data from a relational database perspective.
3. Design the database entities and relationships.
4. Prepare normalized tables for MySQL.
5. Create primary and foreign key relationships.
6. Export SQL-ready datasets.
7. Load the data into the `leadership_books_db` MySQL database.
8. Validate database integrity using SQL queries.
9. Construct the final EER diagram.
10. Perform SQL-based analytical queries for later EDA and recommendation-system development.

### Final Integrated Data

The integration stage produced:

- 2,066 unique book entities.
- 950 Open Library work records.
- 1,120 canonical LeadershipNow edition/source records.
- 4 confirmed cross-source linked entities.
- 0 duplicate project book identifiers.
- 0 missing project book identifiers.

The database design uses the project-specific `book_id` as the central identifier while preserving source-specific identifiers and metadata.

In [1]:
import sys

print(sys.executable)
print(sys.version)

/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/.venv/bin/python
3.13.11 | packaged by Anaconda, Inc. | (main, Dec 10 2025, 21:21:08) [Clang 20.1.8 ]


In [2]:
import sys

!{sys.executable} -m pip install sqlalchemy pymysql

In [3]:
import ast
import numpy as np
import pandas as pd

from pathlib import Path

from sqlalchemy import create_engine, text

In [4]:
from sqlalchemy import create_engine, text

print("SQLAlchemy import successful.")

SQLAlchemy import successful.


In [5]:
FINAL_DIR = Path("../data/final")
SQL_DIR = Path("../sql")

SQL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Final data directory:", FINAL_DIR.resolve())
print("SQL directory:", SQL_DIR.resolve())

Final data directory: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/final
SQL directory: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/sql


In [6]:
books = pd.read_csv(
    FINAL_DIR / "books_master.csv"
)

leadershipnow_editions = pd.read_csv(
    FINAL_DIR / "leadershipnow_editions.csv",
    dtype={
        "isbn_source": "string",
        "isbn_normalized": "string",
        "isbn_13": "string"
    }
)

openlibrary = pd.read_csv(
    FINAL_DIR / "openlibrary_integrated.csv"
)

print("Books:", books.shape)
print("LeadershipNow editions:", leadershipnow_editions.shape)
print("Open Library:", openlibrary.shape)

Books: (2067, 18)
LeadershipNow editions: (1120, 23)
Open Library: (950, 50)


In [139]:
# =====================================================
# FILE PATHS
# =====================================================

final_dir = Path("../data/final")

books_path = final_dir / "books_master.csv"
editions_path = final_dir / "leadershipnow_editions.csv"
openlibrary_path = final_dir / "openlibrary_integrated.csv"

print("Books file exists:", books_path.exists())
print("Editions file exists:", editions_path.exists())
print("Open Library file exists:", openlibrary_path.exists())

Books file exists: True
Editions file exists: True
Open Library file exists: True


In [140]:
# =====================================================
# LOAD CORRECTED INTEGRATED DATA
# =====================================================

books = pd.read_csv(books_path)

leadershipnow_editions = pd.read_csv(editions_path)

openlibrary_integrated = pd.read_csv(openlibrary_path)

print("books:", books.shape)
print("leadershipnow_editions:", leadershipnow_editions.shape)
print("openlibrary_integrated:", openlibrary_integrated.shape)

books: (2067, 18)
leadershipnow_editions: (1120, 23)
openlibrary_integrated: (950, 50)


In [141]:
# =====================================================
# SOURCE DATA VALIDATION
# =====================================================

print("Books:", len(books))
print("Unique book IDs:", books["book_id"].nunique())
print("Duplicate book IDs:", books["book_id"].duplicated().sum())

print(
    "\nLeadershipNow editions:",
    len(leadershipnow_editions)
)

print(
    "Valid edition → book relationships:",
    leadershipnow_editions["book_id"]
    .isin(books["book_id"])
    .sum()
)

print(
    "Confirmed cross-source editions:",
    leadershipnow_editions["openlibrary_key"]
    .notna()
    .sum()
)

Books: 2067
Unique book IDs: 2067
Duplicate book IDs: 0

LeadershipNow editions: 1120
Valid edition → book relationships: 1120
Confirmed cross-source editions: 3


In [142]:
# =====================================================
# INSPECT EXISTING SQL-READY FILES
# =====================================================

sql_ready_dir = Path("../data/sql_ready")

sql_ready_files = sorted(
    sql_ready_dir.glob("*.csv")
)

print(
    "Existing SQL-ready files:",
    len(sql_ready_files)
)

for file in sql_ready_files:
    print(file.name)

Existing SQL-ready files: 14
authors.csv
book_authors.csv
book_categories.csv
book_editions.csv
book_metrics.csv
books.csv
categories.csv
data_sources.csv
foreign_key_validation.csv
junction_table_validation.csv
primary_key_validation.csv
relational_table_summary.csv
source_records.csv
sql_export_validation.csv


In [143]:
def parse_list(value):

    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)

            if isinstance(parsed, list):
                return parsed

        except (ValueError, SyntaxError):
            pass

    return []

In [144]:
for column in [
    "authors",
    "subjects"
]:
    books[column] = (
        books[column]
        .apply(parse_list)
    )

In [145]:
openlibrary_list_columns = [
    "authors",
    "author_keys",
    "publish_dates",
    "publishers",
    "isbn_10",
    "isbn_13",
    "all_isbns",
    "languages",
    "subjects",
    "collection_queries",
    "work_subjects"
]

for column in openlibrary_list_columns:

    if column in openlibrary.columns:
        openlibrary[column] = (
            openlibrary[column]
            .apply(parse_list)
        )

In [146]:
print(
    "Books authors type:",
    type(books.loc[0, "authors"])
)

print(
    "Books subjects type:",
    type(books.loc[0, "subjects"])
)

print(
    "Open Library ISBN-13 type:",
    type(openlibrary.loc[0, "isbn_13"])
)

Books authors type: <class 'list'>
Books subjects type: <class 'list'>
Open Library ISBN-13 type: <class 'list'>


In [147]:
validation_summary = pd.DataFrame({
    "check": [
        "Book rows",
        "Unique book IDs",
        "Duplicate book IDs",
        "Missing book IDs",
        "LeadershipNow edition rows",
        "Open Library rows"
    ],
    "result": [
        len(books),
        books["book_id"].nunique(),
        books["book_id"].duplicated().sum(),
        books["book_id"].isna().sum(),
        len(leadershipnow_editions),
        len(openlibrary)
    ]
})

validation_summary

,check,result
0,Book rows,2067
1,Unique book IDs,2067
2,Duplicate book IDs,0
3,Missing book IDs,0
4,LeadershipNow edition rows,1120
5,Open Library rows,950


## Relational Database Design

The integrated dataset contains book-level attributes as well as several one-to-many and many-to-many relationships. Storing all of these values in a single SQL table would introduce repeated values and make relational analysis difficult.

The database is therefore normalized into separate entities.

### Core Relationships

- One book may have one or more authors.
- One author may be associated with multiple books.
- One book may belong to multiple categories.
- One category may describe multiple books.
- One book may have edition-level metadata.
- One book may have reading and rating metrics.
- One book may be represented by records from multiple data sources.

The `books` table serves as the central entity and uses the project-specific `book_id` as its primary key.

Many-to-many relationships are represented using junction tables such as `book_authors` and `book_categories`.

In [148]:
sql_books = books[
    [
        "book_id",
        "canonical_title",
        "description",
        "first_publish_year",
        "publication_year_observed",
        "cover_url",
        "openlibrary_key",
        "source_openlibrary",
        "source_leadershipnow"
    ]
].copy()

In [149]:
sql_books.rename(
    columns={
        "canonical_title": "title"
    },
    inplace=True
)

In [150]:
sql_books.head()

,book_id,title,description,first_publish_year,publication_year_observed,cover_url,openlibrary_key,source_openlibrary,source_leadershipnow
0,BOOK00001,Principle-Centered Leadership,How do we as individuals and organizations sur...,1989.0,NaN,https://covers.openlibrary.org/b/id/10858615-L...,/works/OL2630041W,True,False
1,BOOK00002,Leadership in Organizations,NaN,1981.0,NaN,https://covers.openlibrary.org/b/id/87719-L.jpg,/works/OL2731767W,True,False
2,BOOK00003,Kepemimpinan =,NaN,1977.0,NaN,https://covers.openlibrary.org/b/id/14420782-L...,/works/OL302757W,True,False
3,BOOK00004,Spiritual leadership,NaN,1967.0,NaN,https://covers.openlibrary.org/b/id/570509-L.jpg,/works/OL450702W,True,False
4,BOOK00005,Leadership,NaN,1997.0,NaN,https://covers.openlibrary.org/b/id/3859675-L.jpg,/works/OL94176W,True,False


In [151]:
print(
    "SQL book rows:",
    len(sql_books)
)

print(
    "Unique book IDs:",
    sql_books["book_id"].nunique()
)

print(
    "Duplicate book IDs:",
    sql_books["book_id"].duplicated().sum()
)

print(
    "Missing book IDs:",
    sql_books["book_id"].isna().sum()
)

print(
    "Missing titles:",
    sql_books["title"].isna().sum()
)

SQL book rows: 2067
Unique book IDs: 2067
Duplicate book IDs: 0
Missing book IDs: 0
Missing titles: 0


In [152]:
author_counts = (
    books["authors"]
    .apply(len)
)

author_counts.describe()

count    2067.000000
mean        1.250605
std         0.769766
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        11.000000
Name: authors, dtype: float64

In [153]:
print(
    "Books with no author:",
    (author_counts == 0).sum()
)

print(
    "Books with one author:",
    (author_counts == 1).sum()
)

print(
    "Books with multiple authors:",
    (author_counts > 1).sum()
)

print(
    "Maximum authors on one book:",
    author_counts.max()
)

Books with no author: 8
Books with one author: 1764
Books with multiple authors: 295
Maximum authors on one book: 11


In [154]:
book_author_exploded = (
    books[
        [
            "book_id",
            "authors"
        ]
    ]
    .explode(
        "authors"
    )
    .rename(
        columns={
            "authors": "author_name"
        }
    )
)

In [155]:
book_author_exploded[
    "author_name"
] = (
    book_author_exploded[
        "author_name"
    ]
    .astype("string")
    .str.strip()
)

book_author_exploded = (
    book_author_exploded[
        book_author_exploded[
            "author_name"
        ].notna()
        &
        (
            book_author_exploded[
                "author_name"
            ] != ""
        )
    ]
    .copy()
)

In [156]:
print(
    "Book-author relationships:",
    len(book_author_exploded)
)

print(
    "Unique author names:",
    book_author_exploded[
        "author_name"
    ].nunique()
)

Book-author relationships: 2585
Unique author names: 2289


In [157]:
def normalize_author(value):

    if pd.isna(value):
        return ""

    value = str(value).lower()
    value = value.replace("’", "'")

    value = re.sub(
        r"[^a-z0-9\s]",
        " ",
        value
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()

In [158]:
import re

In [159]:
book_author_exploded[
    "author_normalized"
] = (
    book_author_exploded[
        "author_name"
    ]
    .apply(
        normalize_author
    )
)

In [160]:
book_author_exploded[
    [
        "author_name",
        "author_normalized"
    ]
].head(20)

,author_name,author_normalized
0,Stephen R. Covey,stephen r covey
1,Gary A. Yukl,gary a yukl
2,Karjadi M.,karjadi m
3,J. Oswald Sanders,j oswald sanders
4,Peter Guy Northouse,peter guy northouse
5,John C. Maxwell,john c maxwell
6,Peter G. Northouse,peter g northouse
7,Bernard M. Bass,bernard m bass
8,James Comey,james comey
8,James B. Comey,james b comey


In [161]:
author_normalization_audit = (
    book_author_exploded
    .groupby(
        "author_normalized"
    )
    .agg(
        unique_display_names=(
            "author_name",
            "nunique"
        ),
        display_names=(
            "author_name",
            lambda x:
                sorted(
                    set(x)
                )
        )
    )
    .reset_index()
)

In [162]:
author_name_variants = (
    author_normalization_audit[
        author_normalization_audit[
            "unique_display_names"
        ] > 1
    ]
    .copy()
)

print(
    "Normalized author names with multiple display variants:",
    len(author_name_variants)
)

author_name_variants.head(20)

Normalized author names with multiple display variants: 8


,author_normalized,unique_display_names,display_names
6,abraham holtzman,2,"[Abraham Holtzman, abraham holtzman]"
438,david a whetten,2,"[DAVID A. WHETTEN, David A. Whetten]"
611,eric masinde aseka,2,"[ERIC MASINDE ASEKA, Eric Masinde Aseka]"
723,gordon j curphy,2,"[Gordon J Curphy, Gordon J. Curphy]"
863,james frederick bender,2,"[JAMES FREDERICK BENDER, James Frederick Bender]"
968,jerrold s greenberg,2,"[Jerrold S Greenberg, Jerrold S. Greenberg]"
1239,l m prasad,2,"[L. M. Prasad, L.M. Prasad]"
1888,ronald adler,2,"[RONALD ADLER, Ronald Adler]"


In [163]:
sql_authors = (
    book_author_exploded[
        [
            "author_name",
            "author_normalized"
        ]
    ]
    .drop_duplicates(
        subset=[
            "author_normalized"
        ]
    )
    .sort_values(
        "author_normalized"
    )
    .reset_index(
        drop=True
    )
)

In [164]:
sql_authors[
    "author_id"
] = [
    f"AUTHOR{i:05d}"
    for i in range(
        1,
        len(sql_authors) + 1
    )
]

In [165]:
sql_authors = sql_authors[
    [
        "author_id",
        "author_name",
        "author_normalized"
    ]
]

In [166]:
sql_authors.head(10)

,author_id,author_name,author_normalized
0,AUTHOR00001,A. Brem,a brem
1,AUTHOR00002,A. J. Strickland,a j strickland
2,AUTHOR00003,A. S. Kohli,a s kohli
3,AUTHOR00004,Aaron Antonovsky,aaron antonovsky
4,AUTHOR00005,Aaron Salko,aaron salko
5,AUTHOR00006,Abdullahi Sheikh Ali,abdullahi sheikh ali
6,AUTHOR00007,abraham holtzman,abraham holtzman
7,AUTHOR00008,Adam Alter,adam alter
8,AUTHOR00009,Adam Bryant,adam bryant
9,AUTHOR00010,Adam Christing,adam christing


In [167]:
print(
    "Author rows:",
    len(sql_authors)
)

print(
    "Unique author IDs:",
    sql_authors[
        "author_id"
    ].nunique()
)

print(
    "Duplicate author IDs:",
    sql_authors[
        "author_id"
    ].duplicated().sum()
)

print(
    "Duplicate normalized authors:",
    sql_authors[
        "author_normalized"
    ].duplicated().sum()
)

Author rows: 2281
Unique author IDs: 2281
Duplicate author IDs: 0
Duplicate normalized authors: 0


In [168]:
book_authors = (
    book_author_exploded
    .merge(
        sql_authors[
            [
                "author_id",
                "author_normalized"
            ]
        ],
        on="author_normalized",
        how="left",
        validate="many_to_one"
    )
)

In [169]:
sql_book_authors = (
    book_authors[
        [
            "book_id",
            "author_id"
        ]
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)

In [170]:
sql_book_authors.head(20)

,book_id,author_id
0,BOOK00001,AUTHOR02055
1,BOOK00002,AUTHOR00679
2,BOOK00003,AUTHOR01157
3,BOOK00004,AUTHOR00828
4,BOOK00005,AUTHOR01667
5,BOOK00006,AUTHOR01030
6,BOOK00007,AUTHOR01665
7,BOOK00008,AUTHOR00189
8,BOOK00009,AUTHOR00862
9,BOOK00009,AUTHOR00855


In [171]:
print(
    "Book-author relationships:",
    len(sql_book_authors)
)

print(
    "Missing book IDs:",
    sql_book_authors[
        "book_id"
    ].isna().sum()
)

print(
    "Missing author IDs:",
    sql_book_authors[
        "author_id"
    ].isna().sum()
)

print(
    "Duplicate relationships:",
    sql_book_authors.duplicated().sum()
)

Book-author relationships: 2566
Missing book IDs: 0
Missing author IDs: 0
Duplicate relationships: 0


In [172]:
invalid_book_fk = (
    set(
        sql_book_authors[
            "book_id"
        ]
    )
    -
    set(
        sql_books[
            "book_id"
        ]
    )
)

print(
    "Invalid book foreign keys:",
    len(invalid_book_fk)
)

Invalid book foreign keys: 0


In [173]:
invalid_author_fk = (
    set(
        sql_book_authors[
            "author_id"
        ]
    )
    -
    set(
        sql_authors[
            "author_id"
        ]
    )
)

print(
    "Invalid author foreign keys:",
    len(invalid_author_fk)
)

Invalid author foreign keys: 0


In [174]:
books_with_authors = (
    sql_book_authors[
        "book_id"
    ].nunique()
)

books_without_authors = (
    len(sql_books)
    -
    books_with_authors
)

print(
    "Books with at least one author:",
    books_with_authors
)

print(
    "Books without an identified author:",
    books_without_authors
)

print(
    "Author coverage:",
    round(
        books_with_authors
        / len(sql_books)
        * 100,
        2
    ),
    "%"
)

Books with at least one author: 2059
Books without an identified author: 8
Author coverage: 99.61 %


In [175]:
category_counts = (
    books["subjects"]
    .apply(len)
)

category_counts.describe()

count    2067.000000
mean        2.895017
std         5.814918
min         0.000000
25%         0.000000
50%         0.000000
75%         3.000000
max        59.000000
Name: subjects, dtype: float64

In [176]:
print(
    "Books with no categories:",
    (category_counts == 0).sum()
)

print(
    "Books with one category:",
    (category_counts == 1).sum()
)

print(
    "Books with multiple categories:",
    (category_counts > 1).sum()
)

print(
    "Maximum categories on one book:",
    category_counts.max()
)

Books with no categories: 1241
Books with one category: 118
Books with multiple categories: 708
Maximum categories on one book: 59


In [177]:
book_category_exploded = (
    books[
        [
            "book_id",
            "subjects"
        ]
    ]
    .explode(
        "subjects"
    )
    .rename(
        columns={
            "subjects": "category_name"
        }
    )
)

In [178]:
book_category_exploded[
    "category_name"
] = (
    book_category_exploded[
        "category_name"
    ]
    .astype("string")
    .str.strip()
)

book_category_exploded = (
    book_category_exploded[
        book_category_exploded[
            "category_name"
        ].notna()
        &
        (
            book_category_exploded[
                "category_name"
            ] != ""
        )
    ]
    .copy()
)

In [179]:
print(
    "Book-category relationships before normalization:",
    len(book_category_exploded)
)

print(
    "Unique category labels before normalization:",
    book_category_exploded[
        "category_name"
    ].nunique()
)

Book-category relationships before normalization: 5984
Unique category labels before normalization: 2300


In [180]:
def normalize_category(value):

    if pd.isna(value):
        return ""

    value = str(value).lower()

    value = value.replace(
        "’",
        "'"
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()

In [181]:
book_category_exploded[
    "category_normalized"
] = (
    book_category_exploded[
        "category_name"
    ]
    .apply(
        normalize_category
    )
)

In [182]:
book_category_exploded[
    [
        "category_name",
        "category_normalized"
    ]
].head(20)

,category_name,category_normalized
0,Leadership,leadership
0,Psychological aspects of Success,psychological aspects of success
0,Success,success
0,Psychological aspects,psychological aspects
0,Commerce,commerce
0,Success in business,success in business
0,Aptitude pour la direction,aptitude pour la direction
0,Achievement,achievement
0,Success--,success--
1,Organisation,organisation


In [183]:
category_normalization_audit = (
    book_category_exploded
    .groupby(
        "category_normalized"
    )
    .agg(
        unique_display_names=(
            "category_name",
            "nunique"
        ),
        display_names=(
            "category_name",
            lambda x:
                sorted(
                    set(x)
                )
        )
    )
    .reset_index()
)

In [184]:
category_name_variants = (
    category_normalization_audit[
        category_normalization_audit[
            "unique_display_names"
        ] > 1
    ]
    .copy()
)

print(
    "Normalized categories with multiple display variants:",
    len(category_name_variants)
)

category_name_variants.head(20)

Normalized categories with multiple display variants: 105


,category_normalized,unique_display_names,display_names
75,administracao da producao,2,"[Administracao Da Producao, Administracao da p..."
122,alienation (social psychology),2,"[Alienation (Social psychology), Alienation (s..."
177,atrial fibrillation,2,"[Atrial Fibrillation, Atrial fibrillation]"
179,"attentats du 11 septembre 2001, e tats-unis",2,"[Attentats du 11 septembre 2001, E tats-Unis, ..."
180,"attentats du 11 septembre 2001, états-unis",2,"[Attentats du 11 septembre 2001, États-Unis, A..."
230,biography,2,"[Biography, biography]"
250,business & economics,3,"[BUSINESS & ECONOMICS, Business & Economics, B..."
265,business & economics / economics / general,2,"[BUSINESS & ECONOMICS / Economics / General, B..."
270,business & economics / human resources & perso...,2,[BUSINESS & ECONOMICS / Human Resources & Pers...
275,business & economics / leadership,2,"[BUSINESS & ECONOMICS / Leadership, Business &..."


In [185]:
sql_categories = (
    book_category_exploded[
        [
            "category_name",
            "category_normalized"
        ]
    ]
    .drop_duplicates(
        subset=[
            "category_normalized"
        ]
    )
    .sort_values(
        "category_normalized"
    )
    .reset_index(
        drop=True
    )
)

In [186]:
sql_categories[
    "category_id"
] = [
    f"CAT{i:05d}"
    for i in range(
        1,
        len(sql_categories) + 1
    )
]

In [187]:
sql_categories = sql_categories[
    [
        "category_id",
        "category_name",
        "category_normalized"
    ]
]

In [188]:
sql_categories.head(10)

,category_id,category_name,category_normalized
0,CAT00001,0 Gesamtdarstellung,0 gesamtdarstellung
1,CAT00002,005.1,005.1
2,CAT00003,152.4,152.4
3,CAT00004,152.4 bra,152.4 bra
4,CAT00005,153.8/3,153.8/3
5,CAT00006,158,158
6,CAT00007,174/.4,174/.4
7,CAT00008,1860 Republican National Convention,1860 republican national convention
8,CAT00009,1864 presidential election,1864 presidential election
9,CAT00010,19th Century,19th century


In [189]:
print(
    "Category rows:",
    len(sql_categories)
)

print(
    "Unique category IDs:",
    sql_categories[
        "category_id"
    ].nunique()
)

print(
    "Duplicate category IDs:",
    sql_categories[
        "category_id"
    ].duplicated().sum()
)

print(
    "Duplicate normalized categories:",
    sql_categories[
        "category_normalized"
    ].duplicated().sum()
)

Category rows: 2187
Unique category IDs: 2187
Duplicate category IDs: 0
Duplicate normalized categories: 0


In [190]:
book_categories = (
    book_category_exploded
    .merge(
        sql_categories[
            [
                "category_id",
                "category_normalized"
            ]
        ],
        on="category_normalized",
        how="left",
        validate="many_to_one"
    )
)

In [191]:
sql_book_categories = (
    book_categories[
        [
            "book_id",
            "category_id"
        ]
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)

In [192]:
sql_book_categories.head(20)

,book_id,category_id
0,BOOK00001,CAT01180
1,BOOK00001,CAT01688
2,BOOK00001,CAT01969
3,BOOK00001,CAT01680
4,BOOK00001,CAT00407
5,BOOK00001,CAT01970
6,BOOK00001,CAT00143
7,BOOK00001,CAT00065
8,BOOK00001,CAT01973
9,BOOK00002,CAT01447


In [193]:
print(
    "Book-category relationships:",
    len(sql_book_categories)
)

print(
    "Missing book IDs:",
    sql_book_categories[
        "book_id"
    ].isna().sum()
)

print(
    "Missing category IDs:",
    sql_book_categories[
        "category_id"
    ].isna().sum()
)

print(
    "Duplicate relationships:",
    sql_book_categories.duplicated().sum()
)

Book-category relationships: 5926
Missing book IDs: 0
Missing category IDs: 0
Duplicate relationships: 0


In [194]:
invalid_category_book_fk = (
    set(
        sql_book_categories[
            "book_id"
        ]
    )
    -
    set(
        sql_books[
            "book_id"
        ]
    )
)

invalid_category_fk = (
    set(
        sql_book_categories[
            "category_id"
        ]
    )
    -
    set(
        sql_categories[
            "category_id"
        ]
    )
)

print(
    "Invalid book foreign keys:",
    len(invalid_category_book_fk)
)

print(
    "Invalid category foreign keys:",
    len(invalid_category_fk)
)

Invalid book foreign keys: 0
Invalid category foreign keys: 0


In [195]:
invalid_category_book_fk = (
    set(
        sql_book_categories[
            "book_id"
        ]
    )
    -
    set(
        sql_books[
            "book_id"
        ]
    )
)

invalid_category_fk = (
    set(
        sql_book_categories[
            "category_id"
        ]
    )
    -
    set(
        sql_categories[
            "category_id"
        ]
    )
)

print(
    "Invalid book foreign keys:",
    len(invalid_category_book_fk)
)

print(
    "Invalid category foreign keys:",
    len(invalid_category_fk)
)

Invalid book foreign keys: 0
Invalid category foreign keys: 0


In [196]:
books_with_categories = (
    sql_book_categories[
        "book_id"
    ].nunique()
)

books_without_categories = (
    len(sql_books)
    -
    books_with_categories
)

print(
    "Books with at least one category:",
    books_with_categories
)

print(
    "Books without categories:",
    books_without_categories
)

print(
    "Category coverage:",
    round(
        books_with_categories
        / len(sql_books)
        * 100,
        2
    ),
    "%"
)

Books with at least one category: 826
Books without categories: 1241
Category coverage: 39.96 %


In [197]:
leadershipnow_editions[
    [
        "book_id",
        "scrape_id",
        "title",
        "isbn_13",
        "isbn_status",
        "publisher",
        "publication_date",
        "publication_year",
        "format",
        "page_count",
        "edition_note"
    ]
].head()

,book_id,scrape_id,title,isbn_13,isbn_status,publisher,publication_date,publication_year,format,page_count,edition_note
0,BOOK00951,306,Crisis Capable,9708891880115,Invalid prefix,Advantage Media Group,2024-10-15,2024,Hardcover,160.0,NaN
1,BOOK00952,733,An Ordinary Man,9780062684165,Valid ISBN-13,Harper,2023-04-11,2023,Hardcover,848.0,NaN
2,BOOK00953,1011,Don't Trust Your Gut,9780062880918,Valid ISBN-13,Dey Street Books,2022-05-10,2022,Hardcover,320.0,NaN
3,BOOK00954,936,Inventor of the Future,9780062947222,Valid ISBN-13,Dey Street Books,2022-08-02,2022,Hardcover,672.0,NaN
4,BOOK00955,1048,The Way Forward,9780062994073,Valid ISBN-13,Dey Street Books,2022-03-01,2022,Hardcover,288.0,NaN


In [198]:
sql_book_editions = leadershipnow_editions[
    [
        "book_id",
        "scrape_id",
        "isbn_source",
        "isbn_normalized",
        "isbn_13",
        "isbn_status",
        "publisher",
        "publication_date",
        "publication_year",
        "format",
        "page_count",
        "edition_note",
        "cover_url",
        "external_book_url",
        "source_metadata_conflict"
    ]
].copy()

In [199]:
sql_book_editions.insert(
    0,
    "edition_id",
    [
        f"EDITION{i:05d}"
        for i in range(
            1,
            len(sql_book_editions) + 1
        )
    ]
)

In [200]:
sql_book_editions.head()

,edition_id,book_id,scrape_id,isbn_source,isbn_normalized,isbn_13,isbn_status,publisher,publication_date,publication_year,format,page_count,edition_note,cover_url,external_book_url,source_metadata_conflict
0,EDITION00001,BOOK00951,306,9708891880115,9708891880115,9708891880115,Invalid prefix,Advantage Media Group,2024-10-15,2024,Hardcover,160.0,NaN,https://www.leadershipnow.com/leadershop/image...,https://amzn.to/3NSXJFb,False
1,EDITION00002,BOOK00952,733,9780062684165,9780062684165,9780062684165,Valid ISBN-13,Harper,2023-04-11,2023,Hardcover,848.0,NaN,https://www.leadershipnow.com/leadershop/image...,https://amzn.to/3rIwC4M,False
2,EDITION00003,BOOK00953,1011,9780062880918,9780062880918,9780062880918,Valid ISBN-13,Dey Street Books,2022-05-10,2022,Hardcover,320.0,NaN,https://www.leadershipnow.com/leadershop/image...,https://amzn.to/3F6nrAX,False
3,EDITION00004,BOOK00954,936,9780062947222,9780062947222,9780062947222,Valid ISBN-13,Dey Street Books,2022-08-02,2022,Hardcover,672.0,NaN,https://www.leadershipnow.com/leadershop/image...,https://amzn.to/3AYOj4a,False
4,EDITION00005,BOOK00955,1048,9780062994073,9780062994073,9780062994073,Valid ISBN-13,Dey Street Books,2022-03-01,2022,Hardcover,288.0,NaN,https://www.leadershipnow.com/leadershop/image...,https://amzn.to/3LbsKl2,False


In [201]:
print(
    "Edition rows:",
    len(sql_book_editions)
)

print(
    "Unique edition IDs:",
    sql_book_editions[
        "edition_id"
    ].nunique()
)

print(
    "Duplicate edition IDs:",
    sql_book_editions[
        "edition_id"
    ].duplicated().sum()
)

print(
    "Missing book IDs:",
    sql_book_editions[
        "book_id"
    ].isna().sum()
)

print(
    "Duplicate scrape IDs:",
    sql_book_editions[
        "scrape_id"
    ].duplicated().sum()
)

Edition rows: 1120
Unique edition IDs: 1120
Duplicate edition IDs: 0
Missing book IDs: 0
Duplicate scrape IDs: 0


In [202]:
invalid_edition_book_fk = (
    set(
        sql_book_editions[
            "book_id"
        ]
    )
    -
    set(
        sql_books[
            "book_id"
        ]
    )
)

print(
    "Invalid book foreign keys:",
    len(invalid_edition_book_fk)
)

Invalid book foreign keys: 0


In [203]:
isbn_status_summary = (
    sql_book_editions[
        "isbn_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "isbn_status"
    )
    .reset_index(
        name="editions"
    )
)

isbn_status_summary

,isbn_status,editions
0,Valid ISBN-13,1057
1,Invalid check digit,57
2,Invalid length,3
3,Invalid characters,2
4,Invalid prefix,1


In [204]:
isbn_length_summary = (
    sql_book_editions[
        "isbn_normalized"
    ]
    .str.len()
    .value_counts(
        dropna=False
    )
    .sort_index()
)

isbn_length_summary

isbn_normalized
12       2
13    1117
14       1
Name: count, dtype: int64

In [205]:
edition_fields = [
    "isbn_normalized",
    "publisher",
    "publication_date",
    "publication_year",
    "format",
    "page_count",
    "edition_note",
    "cover_url",
    "external_book_url"
]

edition_coverage = pd.DataFrame({
    "field": edition_fields,
    "available": [
        sql_book_editions[
            column
        ].notna().sum()
        for column in edition_fields
    ]
})

edition_coverage[
    "coverage_pct"
] = (
    edition_coverage[
        "available"
    ]
    /
    len(sql_book_editions)
    *
    100
).round(2)

edition_coverage

,field,available,coverage_pct
0,isbn_normalized,1120,100.00
1,publisher,1120,100.00
2,publication_date,1120,100.00
3,publication_year,1120,100.00
4,format,1120,100.00
5,page_count,1119,99.91
6,edition_note,4,0.36
7,cover_url,1120,100.00
8,external_book_url,1120,100.00


In [206]:
books_with_editions = (
    sql_book_editions[
        "book_id"
    ].nunique()
)

print(
    "Books represented by LeadershipNow editions:",
    books_with_editions
)

print(
    "Books without LeadershipNow edition records:",
    len(sql_books)
    -
    books_with_editions
)

print(
    "Edition coverage:",
    round(
        books_with_editions
        / len(sql_books)
        * 100,
        2
    ),
    "%"
)

Books represented by LeadershipNow editions: 1120
Books without LeadershipNow edition records: 947
Edition coverage: 54.18 %


In [207]:
sql_book_metrics = books[
    [
        "book_id",
        "average_rating",
        "ratings_count",
        "edition_count",
        "want_to_read_count",
        "currently_reading_count",
        "already_read_count"
    ]
].copy()

In [208]:
metric_columns = [
    "average_rating",
    "ratings_count",
    "edition_count",
    "want_to_read_count",
    "currently_reading_count",
    "already_read_count"
]

sql_book_metrics = (
    sql_book_metrics[
        sql_book_metrics[
            metric_columns
        ]
        .notna()
        .any(axis=1)
    ]
    .copy()
)

In [209]:
sql_book_metrics.head()

,book_id,average_rating,ratings_count,edition_count,want_to_read_count,currently_reading_count,already_read_count
0,BOOK00001,4.500000,2.0,21.0,145.0,15.0,6.0
1,BOOK00002,5.000000,1.0,26.0,110.0,12.0,0.0
2,BOOK00003,1.000000,1.0,1.0,41.0,2.0,18.0
3,BOOK00004,5.000000,3.0,8.0,74.0,7.0,2.0
4,BOOK00005,3.666667,3.0,5.0,67.0,4.0,3.0


In [210]:
print(
    "Metric rows:",
    len(sql_book_metrics)
)

print(
    "Unique book IDs:",
    sql_book_metrics[
        "book_id"
    ].nunique()
)

print(
    "Duplicate book IDs:",
    sql_book_metrics[
        "book_id"
    ].duplicated().sum()
)

print(
    "Missing book IDs:",
    sql_book_metrics[
        "book_id"
    ].isna().sum()
)

Metric rows: 950
Unique book IDs: 950
Duplicate book IDs: 0
Missing book IDs: 0


In [211]:
invalid_metric_book_fk = (
    set(
        sql_book_metrics[
            "book_id"
        ]
    )
    -
    set(
        sql_books[
            "book_id"
        ]
    )
)

print(
    "Invalid book foreign keys:",
    len(invalid_metric_book_fk)
)

Invalid book foreign keys: 0


In [212]:
metrics_coverage = pd.DataFrame({
    "metric": metric_columns,
    "available": [
        sql_book_metrics[
            column
        ].notna().sum()
        for column in metric_columns
    ]
})

metrics_coverage[
    "coverage_pct_of_all_books"
] = (
    metrics_coverage[
        "available"
    ]
    /
    len(sql_books)
    *
    100
).round(2)

metrics_coverage[
    "coverage_pct_of_metric_books"
] = (
    metrics_coverage[
        "available"
    ]
    /
    len(sql_book_metrics)
    *
    100
).round(2)

metrics_coverage

,metric,available,coverage_pct_of_all_books,coverage_pct_of_metric_books
0,average_rating,282,13.64,29.68
1,ratings_count,282,13.64,29.68
2,edition_count,950,45.96,100.00
3,want_to_read_count,807,39.04,84.95
4,currently_reading_count,807,39.04,84.95
5,already_read_count,807,39.04,84.95


In [213]:
print(
    "Minimum rating:",
    sql_book_metrics[
        "average_rating"
    ].min()
)

print(
    "Maximum rating:",
    sql_book_metrics[
        "average_rating"
    ].max()
)

print(
    "Negative rating counts:",
    (
        sql_book_metrics[
            "ratings_count"
        ] < 0
    ).sum()
)

print(
    "Negative edition counts:",
    (
        sql_book_metrics[
            "edition_count"
        ] < 0
    ).sum()
)

Minimum rating: 0.0
Maximum rating: 5.0
Negative rating counts: 0
Negative edition counts: 0


## Data Sources and Source Provenance

The project combines information collected through two different acquisition methods:

1. Open Library — API-based collection of work-level bibliographic and engagement metadata.
2. LeadershipNow / LeaderShop — web-based collection of edition and release metadata.

A dedicated `data_sources` table identifies the origin and collection method of each source.

A separate `source_records` table connects individual source records to the project's canonical `book_id`.

This preserves data lineage and allows the database to distinguish between a canonical book entity and the original records from which that entity was constructed.

In [214]:
sql_data_sources = pd.DataFrame({
    "data_source_id": [
        "SRC001",
        "SRC002"
    ],
    "source_name": [
        "Open Library",
        "LeadershipNow / LeaderShop"
    ],
    "source_type": [
        "API",
        "Web"
    ],
    "data_level": [
        "Work-level",
        "Edition-level"
    ]
})

sql_data_sources

,data_source_id,source_name,source_type,data_level
0,SRC001,Open Library,API,Work-level
1,SRC002,LeadershipNow / LeaderShop,Web,Edition-level


In [215]:
print(
    "Data source rows:",
    len(sql_data_sources)
)

print(
    "Unique source IDs:",
    sql_data_sources[
        "data_source_id"
    ].nunique()
)

print(
    "Duplicate source IDs:",
    sql_data_sources[
        "data_source_id"
    ].duplicated().sum()
)

Data source rows: 2
Unique source IDs: 2
Duplicate source IDs: 0


In [216]:
openlibrary[
    [
        column
        for column in [
            "book_id",
            "openlibrary_key",
            "source_url",
            "collection_queries",
            "query_match_count"
        ]
        if column in openlibrary.columns
    ]
].head()

,book_id,openlibrary_key,source_url,collection_queries,query_match_count
0,BOOK00001,/works/OL2630041W,https://openlibrary.org/works/OL2630041W,[leadership],1
1,BOOK00002,/works/OL2731767W,https://openlibrary.org/works/OL2731767W,"[leadership, leadership development, executive...",6
2,BOOK00003,/works/OL302757W,https://openlibrary.org/works/OL302757W,[leadership],1
3,BOOK00004,/works/OL450702W,https://openlibrary.org/works/OL450702W,[leadership],1
4,BOOK00005,/works/OL94176W,https://openlibrary.org/works/OL94176W,[leadership],1


In [217]:
[
    column
    for column in openlibrary.columns
    if (
        "source" in column.lower()
        or "query" in column.lower()
        or "url" in column.lower()
        or "key" in column.lower()
    )
]

['openlibrary_key',
 'author_keys',
 'cover_url',
 'source',
 'source_url',
 'query_match_count',
 'source_openlibrary',
 'source_leadershipnow',
 'integration_source']

In [218]:
leadershipnow_editions[
    [
        column
        for column in [
            "book_id",
            "scrape_id",
            "release_page",
            "release_page_url",
            "external_book_url",
            "match_method",
            "match_confidence"
        ]
        if column in leadershipnow_editions.columns
    ]
].head()

,book_id,scrape_id,external_book_url,match_method,match_confidence
0,BOOK00951,306,https://amzn.to/3NSXJFb,NaN,NaN
1,BOOK00952,733,https://amzn.to/3rIwC4M,NaN,NaN
2,BOOK00953,1011,https://amzn.to/3F6nrAX,NaN,NaN
3,BOOK00954,936,https://amzn.to/3AYOj4a,NaN,NaN
4,BOOK00955,1048,https://amzn.to/3LbsKl2,NaN,NaN


In [219]:
leadershipnow_editions[
    [
        column
        for column in [
            "book_id",
            "scrape_id",
            "release_page",
            "release_page_url",
            "external_book_url",
            "match_method",
            "match_confidence"
        ]
        if column in leadershipnow_editions.columns
    ]
].head()

,book_id,scrape_id,external_book_url,match_method,match_confidence
0,BOOK00951,306,https://amzn.to/3NSXJFb,NaN,NaN
1,BOOK00952,733,https://amzn.to/3rIwC4M,NaN,NaN
2,BOOK00953,1011,https://amzn.to/3F6nrAX,NaN,NaN
3,BOOK00954,936,https://amzn.to/3AYOj4a,NaN,NaN
4,BOOK00955,1048,https://amzn.to/3LbsKl2,NaN,NaN


In [220]:
[
    column
    for column in leadershipnow_editions.columns
    if (
        "source" in column.lower()
        or "release" in column.lower()
        or "match" in column.lower()
        or "url" in column.lower()
        or "scrape" in column.lower()
    )
]

['scrape_id',
 'isbn_source',
 'cover_url',
 'external_book_url',
 'release_pages',
 'source_metadata_conflict',
 'source',
 'match_method',
 'match_confidence']

In [221]:
leadershipnow_editions[
    [
        "scrape_id",
        "release_pages"
    ]
].head(10)

,scrape_id,release_pages
0,306,['2024 Releases']
1,733,['2023 Releases']
2,1011,['2022 Releases']
3,936,['2022 Releases']
4,1048,['2022 Releases']
5,1113,['2022 Releases']
6,998,['2022 Releases']
7,758,['2023 Releases']
8,1001,['2022 Releases']
9,993,['2022 Releases']


In [222]:
print(
    "release_pages type:",
    type(
        leadershipnow_editions.loc[
            0,
            "release_pages"
        ]
    )
)

print(
    "Example:",
    leadershipnow_editions.loc[
        0,
        "release_pages"
    ]
)

release_pages type: <class 'str'>
Example: ['2024 Releases']


In [223]:
leadershipnow_editions[
    "release_pages"
] = (
    leadershipnow_editions[
        "release_pages"
    ]
    .apply(parse_list)
)

In [224]:
print(
    type(
        leadershipnow_editions.loc[
            0,
            "release_pages"
        ]
    )
)

leadershipnow_editions[
    [
        "scrape_id",
        "release_pages"
    ]
].head()

<class 'list'>


,scrape_id,release_pages
0,306,[2024 Releases]
1,733,[2023 Releases]
2,1011,[2022 Releases]
3,936,[2022 Releases]
4,1048,[2022 Releases]


In [225]:
ol_source_records = openlibrary[
    [
        "book_id",
        "openlibrary_key",
        "source_url",
        "query_match_count"
    ]
].copy()

In [226]:
ol_source_records.rename(
    columns={
        "openlibrary_key": "source_native_id"
    },
    inplace=True
)

In [227]:
ol_source_records.rename(
    columns={
        "openlibrary_key": "source_native_id"
    },
    inplace=True
)

In [228]:
ol_source_records[
    "data_source_id"
] = "SRC001"

In [229]:
ol_source_records[
    "source_record_type"
] = "Work"

In [230]:
ol_source_records.head()

,book_id,source_native_id,source_url,query_match_count,data_source_id,source_record_type
0,BOOK00001,/works/OL2630041W,https://openlibrary.org/works/OL2630041W,1,SRC001,Work
1,BOOK00002,/works/OL2731767W,https://openlibrary.org/works/OL2731767W,6,SRC001,Work
2,BOOK00003,/works/OL302757W,https://openlibrary.org/works/OL302757W,1,SRC001,Work
3,BOOK00004,/works/OL450702W,https://openlibrary.org/works/OL450702W,1,SRC001,Work
4,BOOK00005,/works/OL94176W,https://openlibrary.org/works/OL94176W,1,SRC001,Work


In [231]:
print(
    "Open Library source records:",
    len(ol_source_records)
)

print(
    "Unique native IDs:",
    ol_source_records[
        "source_native_id"
    ].nunique()
)

print(
    "Missing native IDs:",
    ol_source_records[
        "source_native_id"
    ].isna().sum()
)

print(
    "Duplicate native IDs:",
    ol_source_records[
        "source_native_id"
    ].duplicated().sum()
)

Open Library source records: 950
Unique native IDs: 950
Missing native IDs: 0
Duplicate native IDs: 0


In [232]:
ln_source_records = leadershipnow_editions[
    [
        "book_id",
        "scrape_id",
        "release_pages",
        "match_method",
        "match_confidence"
    ]
].copy()

In [233]:
ln_source_records.rename(
    columns={
        "scrape_id": "source_native_id"
    },
    inplace=True
)

In [234]:
ln_source_records[
    "data_source_id"
] = "SRC002"

ln_source_records[
    "source_record_type"
] = "Edition"

In [235]:
ln_source_records.head()

,book_id,source_native_id,release_pages,match_method,match_confidence,data_source_id,source_record_type
0,BOOK00951,306,[2024 Releases],NaN,NaN,SRC002,Edition
1,BOOK00952,733,[2023 Releases],NaN,NaN,SRC002,Edition
2,BOOK00953,1011,[2022 Releases],NaN,NaN,SRC002,Edition
3,BOOK00954,936,[2022 Releases],NaN,NaN,SRC002,Edition
4,BOOK00955,1048,[2022 Releases],NaN,NaN,SRC002,Edition


In [236]:
print(
    "LeadershipNow source records:",
    len(ln_source_records)
)

print(
    "Unique native IDs:",
    ln_source_records[
        "source_native_id"
    ].nunique()
)

print(
    "Missing native IDs:",
    ln_source_records[
        "source_native_id"
    ].isna().sum()
)

print(
    "Duplicate native IDs:",
    ln_source_records[
        "source_native_id"
    ].duplicated().sum()
)

LeadershipNow source records: 1120
Unique native IDs: 1120
Missing native IDs: 0
Duplicate native IDs: 0


In [237]:
ol_source_records[
    "release_pages"
] = None

ol_source_records[
    "match_method"
] = None

ol_source_records[
    "match_confidence"
] = None

In [238]:
ln_source_records[
    "source_url"
] = None

ln_source_records[
    "query_match_count"
] = None

In [239]:
source_record_columns = [
    "book_id",
    "data_source_id",
    "source_native_id",
    "source_record_type",
    "source_url",
    "query_match_count",
    "release_pages",
    "match_method",
    "match_confidence"
]

ol_source_records = (
    ol_source_records[
        source_record_columns
    ]
)

ln_source_records = (
    ln_source_records[
        source_record_columns
    ]
)

In [240]:
sql_source_records = pd.concat(
    [
        ol_source_records,
        ln_source_records
    ],
    ignore_index=True
)

In [241]:
sql_source_records.insert(
    0,
    "source_record_id",
    [
        f"RECORD{i:05d}"
        for i in range(
            1,
            len(sql_source_records) + 1
        )
    ]
)

In [242]:
sql_source_records.head()

,source_record_id,book_id,data_source_id,source_native_id,source_record_type,source_url,query_match_count,release_pages,match_method,match_confidence
0,RECORD00001,BOOK00001,SRC001,/works/OL2630041W,Work,https://openlibrary.org/works/OL2630041W,1,None,None,None
1,RECORD00002,BOOK00002,SRC001,/works/OL2731767W,Work,https://openlibrary.org/works/OL2731767W,6,None,None,None
2,RECORD00003,BOOK00003,SRC001,/works/OL302757W,Work,https://openlibrary.org/works/OL302757W,1,None,None,None
3,RECORD00004,BOOK00004,SRC001,/works/OL450702W,Work,https://openlibrary.org/works/OL450702W,1,None,None,None
4,RECORD00005,BOOK00005,SRC001,/works/OL94176W,Work,https://openlibrary.org/works/OL94176W,1,None,None,None


In [243]:
print(
    "Source record rows:",
    len(sql_source_records)
)

print(
    "Unique source record IDs:",
    sql_source_records[
        "source_record_id"
    ].nunique()
)

print(
    "Duplicate source record IDs:",
    sql_source_records[
        "source_record_id"
    ].duplicated().sum()
)

print(
    "Missing book IDs:",
    sql_source_records[
        "book_id"
    ].isna().sum()
)

print(
    "Missing source IDs:",
    sql_source_records[
        "data_source_id"
    ].isna().sum()
)

Source record rows: 2070
Unique source record IDs: 2070
Duplicate source record IDs: 0
Missing book IDs: 0
Missing source IDs: 0


In [244]:
duplicate_source_native_keys = (
    sql_source_records[
        [
            "data_source_id",
            "source_native_id"
        ]
    ]
    .duplicated()
    .sum()
)

print(
    "Duplicate source/native ID combinations:",
    duplicate_source_native_keys
)

Duplicate source/native ID combinations: 0


In [245]:
invalid_source_book_fk = (
    set(
        sql_source_records[
            "book_id"
        ]
    )
    -
    set(
        sql_books[
            "book_id"
        ]
    )
)

print(
    "Invalid book foreign keys:",
    len(invalid_source_book_fk)
)

Invalid book foreign keys: 0


In [246]:
invalid_data_source_fk = (
    set(
        sql_source_records[
            "data_source_id"
        ]
    )
    -
    set(
        sql_data_sources[
            "data_source_id"
        ]
    )
)

print(
    "Invalid data-source foreign keys:",
    len(invalid_data_source_fk)
)

Invalid data-source foreign keys: 0


In [247]:
source_distribution = (
    sql_source_records
    .groupby(
        [
            "data_source_id",
            "source_record_type"
        ]
    )
    .size()
    .reset_index(
        name="records"
    )
)

source_distribution

,data_source_id,source_record_type,records
0,SRC001,Work,950
1,SRC002,Edition,1120


In [248]:
book_source_counts = (
    sql_source_records
    .groupby(
        "book_id"
    )[
        "data_source_id"
    ]
    .nunique()
)

print(
    "Books represented by one source:",
    (book_source_counts == 1).sum()
)

print(
    "Books represented by multiple sources:",
    (book_source_counts > 1).sum()
)

print(
    "Maximum sources for one book:",
    book_source_counts.max()
)

Books represented by one source: 2064
Books represented by multiple sources: 3
Maximum sources for one book: 2


## Final Relational Validation

Before exporting the relational tables or loading them into MySQL, the complete relational model is validated.

The validation checks:

- primary-key uniqueness,
- missing primary keys,
- duplicate junction-table relationships,
- foreign-key integrity,
- table row counts,
- source provenance consistency.

Only after these checks pass will the tables be exported for SQL ingestion.

In [249]:
relational_summary = pd.DataFrame({
    "table": [
        "books",
        "authors",
        "book_authors",
        "categories",
        "book_categories",
        "book_editions",
        "book_metrics",
        "data_sources",
        "source_records"
    ],
    "rows": [
        len(sql_books),
        len(sql_authors),
        len(sql_book_authors),
        len(sql_categories),
        len(sql_book_categories),
        len(sql_book_editions),
        len(sql_book_metrics),
        len(sql_data_sources),
        len(sql_source_records)
    ]
})

relational_summary

,table,rows
0,books,2067
1,authors,2281
2,book_authors,2566
3,categories,2187
4,book_categories,5926
5,book_editions,1120
6,book_metrics,950
7,data_sources,2
8,source_records,2070


In [250]:
primary_key_validation = pd.DataFrame({
    "table": [
        "books",
        "authors",
        "categories",
        "book_editions",
        "data_sources",
        "source_records"
    ],
    "primary_key": [
        "book_id",
        "author_id",
        "category_id",
        "edition_id",
        "data_source_id",
        "source_record_id"
    ],
    "duplicate_keys": [
        sql_books["book_id"].duplicated().sum(),
        sql_authors["author_id"].duplicated().sum(),
        sql_categories["category_id"].duplicated().sum(),
        sql_book_editions["edition_id"].duplicated().sum(),
        sql_data_sources["data_source_id"].duplicated().sum(),
        sql_source_records["source_record_id"].duplicated().sum()
    ],
    "missing_keys": [
        sql_books["book_id"].isna().sum(),
        sql_authors["author_id"].isna().sum(),
        sql_categories["category_id"].isna().sum(),
        sql_book_editions["edition_id"].isna().sum(),
        sql_data_sources["data_source_id"].isna().sum(),
        sql_source_records["source_record_id"].isna().sum()
    ]
})

primary_key_validation

,table,primary_key,duplicate_keys,missing_keys
0,books,book_id,0,0
1,authors,author_id,0,0
2,categories,category_id,0,0
3,book_editions,edition_id,0,0
4,data_sources,data_source_id,0,0
5,source_records,source_record_id,0,0


In [251]:
foreign_key_validation = pd.DataFrame({
    "relationship": [
        "book_authors.book_id → books.book_id",
        "book_authors.author_id → authors.author_id",
        "book_categories.book_id → books.book_id",
        "book_categories.category_id → categories.category_id",
        "book_editions.book_id → books.book_id",
        "book_metrics.book_id → books.book_id",
        "source_records.book_id → books.book_id",
        "source_records.data_source_id → data_sources.data_source_id"
    ],
    "invalid_foreign_keys": [
        len(
            set(sql_book_authors["book_id"])
            - set(sql_books["book_id"])
        ),
        len(
            set(sql_book_authors["author_id"])
            - set(sql_authors["author_id"])
        ),
        len(
            set(sql_book_categories["book_id"])
            - set(sql_books["book_id"])
        ),
        len(
            set(sql_book_categories["category_id"])
            - set(sql_categories["category_id"])
        ),
        len(
            set(sql_book_editions["book_id"])
            - set(sql_books["book_id"])
        ),
        len(
            set(sql_book_metrics["book_id"])
            - set(sql_books["book_id"])
        ),
        len(
            set(sql_source_records["book_id"])
            - set(sql_books["book_id"])
        ),
        len(
            set(sql_source_records["data_source_id"])
            - set(sql_data_sources["data_source_id"])
        )
    ]
})

foreign_key_validation

,relationship,invalid_foreign_keys
0,book_authors.book_id → books.book_id,0
1,book_authors.author_id → authors.author_id,0
2,book_categories.book_id → books.book_id,0
3,book_categories.category_id → categories.categ...,0
4,book_editions.book_id → books.book_id,0
5,book_metrics.book_id → books.book_id,0
6,source_records.book_id → books.book_id,0
7,source_records.data_source_id → data_sources.d...,0


In [252]:
junction_validation = pd.DataFrame({
    "table": [
        "book_authors",
        "book_categories"
    ],
    "rows": [
        len(sql_book_authors),
        len(sql_book_categories)
    ],
    "duplicate_relationships": [
        sql_book_authors.duplicated(
            subset=[
                "book_id",
                "author_id"
            ]
        ).sum(),
        sql_book_categories.duplicated(
            subset=[
                "book_id",
                "category_id"
            ]
        ).sum()
    ]
})

junction_validation

,table,rows,duplicate_relationships
0,book_authors,2566,0
1,book_categories,5926,0


In [253]:
all_primary_keys_valid = (
    primary_key_validation[
        "duplicate_keys"
    ].sum() == 0
    and
    primary_key_validation[
        "missing_keys"
    ].sum() == 0
)

all_foreign_keys_valid = (
    foreign_key_validation[
        "invalid_foreign_keys"
    ].sum() == 0
)

all_junctions_valid = (
    junction_validation[
        "duplicate_relationships"
    ].sum() == 0
)

database_ready = (
    all_primary_keys_valid
    and
    all_foreign_keys_valid
    and
    all_junctions_valid
)

print(
    "Primary keys valid:",
    all_primary_keys_valid
)

print(
    "Foreign keys valid:",
    all_foreign_keys_valid
)

print(
    "Junction tables valid:",
    all_junctions_valid
)

print(
    "Database ready:",
    database_ready
)

Primary keys valid: True
Foreign keys valid: True
Junction tables valid: True
Database ready: True


In [254]:
SQL_DATA_DIR = (
    FINAL_DIR.parent
    / "sql_ready"
)

SQL_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "SQL-ready directory:",
    SQL_DATA_DIR.resolve()
)

SQL-ready directory: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/sql_ready


In [255]:
def list_to_text(value):

    if isinstance(value, list):
        return "; ".join(
            str(item)
            for item in value
        )

    if pd.isna(value):
        return None

    return str(value)

In [256]:
sql_source_records[
    "release_pages"
] = (
    sql_source_records[
        "release_pages"
    ]
    .apply(
        list_to_text
    )
)

In [257]:
sql_source_records[
    [
        "source_record_id",
        "data_source_id",
        "release_pages"
    ]
].tail()

,source_record_id,data_source_id,release_pages
2065,RECORD02066,SRC002,2025 Releases
2066,RECORD02067,SRC002,2025 Releases
2067,RECORD02068,SRC002,2025 Releases
2068,RECORD02069,SRC002,2025 Releases
2069,RECORD02070,SRC002,2025 Releases


In [258]:
sql_tables = {
    "books": sql_books,
    "authors": sql_authors,
    "book_authors": sql_book_authors,
    "categories": sql_categories,
    "book_categories": sql_book_categories,
    "book_editions": sql_book_editions,
    "book_metrics": sql_book_metrics,
    "data_sources": sql_data_sources,
    "source_records": sql_source_records
}

In [259]:
for table_name, dataframe in sql_tables.items():

    output_path = (
        SQL_DATA_DIR
        / f"{table_name}.csv"
    )

    dataframe.to_csv(
        output_path,
        index=False
    )

    print(
        f"{table_name}:",
        dataframe.shape,
        "→",
        output_path
    )

books: (2067, 9) → ../data/sql_ready/books.csv
authors: (2281, 3) → ../data/sql_ready/authors.csv
book_authors: (2566, 2) → ../data/sql_ready/book_authors.csv
categories: (2187, 3) → ../data/sql_ready/categories.csv
book_categories: (5926, 2) → ../data/sql_ready/book_categories.csv
book_editions: (1120, 16) → ../data/sql_ready/book_editions.csv
book_metrics: (950, 7) → ../data/sql_ready/book_metrics.csv
data_sources: (2, 4) → ../data/sql_ready/data_sources.csv
source_records: (2070, 10) → ../data/sql_ready/source_records.csv


In [260]:
export_validation = []

for table_name, dataframe in sql_tables.items():

    file_path = (
        SQL_DATA_DIR
        / f"{table_name}.csv"
    )

    reloaded = pd.read_csv(
        file_path
    )

    export_validation.append({
        "table": table_name,
        "original_rows": len(dataframe),
        "reloaded_rows": len(reloaded),
        "row_match": (
            len(dataframe)
            ==
            len(reloaded)
        ),
        "file_exists": (
            file_path.exists()
        )
    })

export_validation = pd.DataFrame(
    export_validation
)

export_validation

,table,original_rows,reloaded_rows,row_match,file_exists
0,books,2067,2067,True,True
1,authors,2281,2281,True,True
2,book_authors,2566,2566,True,True
3,categories,2187,2187,True,True
4,book_categories,5926,5926,True,True
5,book_editions,1120,1120,True,True
6,book_metrics,950,950,True,True
7,data_sources,2,2,True,True
8,source_records,2070,2070,True,True


In [261]:
relational_summary.to_csv(
    SQL_DATA_DIR
    / "relational_table_summary.csv",
    index=False
)

primary_key_validation.to_csv(
    SQL_DATA_DIR
    / "primary_key_validation.csv",
    index=False
)

foreign_key_validation.to_csv(
    SQL_DATA_DIR
    / "foreign_key_validation.csv",
    index=False
)

junction_validation.to_csv(
    SQL_DATA_DIR
    / "junction_table_validation.csv",
    index=False
)

export_validation.to_csv(
    SQL_DATA_DIR
    / "sql_export_validation.csv",
    index=False
)

print(
    "SQL validation reports saved."
)

SQL validation reports saved.


## MySQL Database Implementation

The validated relational model is now implemented in MySQL.

The database is named:

`leadership_books_db`

The implementation uses:

- primary keys for entity integrity,
- composite primary keys for junction tables,
- foreign keys for referential integrity,
- appropriate numeric and text data types,
- nullable fields where source metadata is genuinely unavailable,
- unique constraints where source identifiers must remain unique.

The schema is created before any records are inserted.

## MySQL Connection and Data Loading

The validated relational DataFrames are now loaded into the MySQL database `leadership_books_db`.

The database password is entered securely using `getpass()` and is not stored in the notebook.

In [262]:
from getpass import getpass
from urllib.parse import quote_plus

In [265]:
mysql_user = "root"
mysql_host = "localhost"
mysql_port = 3306
mysql_database = "leadership_books_db"

mysql_password = getpass(
    "Enter MySQL password: "
)

engine = create_engine(
    f"mysql+pymysql://"
    f"{mysql_user}:"
    f"{quote_plus(mysql_password)}"
    f"@{mysql_host}:{mysql_port}/"
    f"{mysql_database}"
)

Enter MySQL password:  ········


In [266]:
with engine.connect() as connection:

    result = connection.execute(
        text(
            "SELECT DATABASE();"
        )
    )

    print(
        "Connected database:",
        result.scalar()
    )

Connected database: leadership_books_db


In [267]:
with engine.connect() as connection:

    result = connection.execute(
        text(
            """
            SHOW TABLES;
            """
        )
    )

    mysql_tables = [
        row[0]
        for row in result
    ]

print(
    "Tables found:",
    len(mysql_tables)
)

mysql_tables

Tables found: 9


['authors',
 'book_authors',
 'book_categories',
 'book_editions',
 'book_metrics',
 'books',
 'categories',
 'data_sources',
 'source_records']

In [270]:
def prepare_for_mysql(df):
    output = df.copy()
    return output.astype(object).where(
        pd.notna(output),
        None
    )

In [271]:
mysql_data = {
    table_name: prepare_for_mysql(dataframe)
    for table_name, dataframe in sql_tables.items()
}

In [272]:
for table_name, dataframe in mysql_data.items():
    print(
        table_name,
        dataframe.shape
    )

books (2067, 9)
authors (2281, 3)
book_authors (2566, 2)
categories (2187, 3)
book_categories (5926, 2)
book_editions (1120, 16)
book_metrics (950, 7)
data_sources (2, 4)
source_records (2070, 10)


In [273]:
with engine.connect() as connection:

    for table_name in sql_tables.keys():

        result = connection.execute(
            text(
                f"SELECT COUNT(*) FROM {table_name};"
            )
        )

        print(
            table_name,
            result.scalar()
        )

books 2066
authors 2280
book_authors 2565
categories 2187
book_categories 5926
book_editions 1120
book_metrics 950
data_sources 2
source_records 2070


In [274]:
leadershipnow_editions[
    "match_confidence"
].value_counts(
    dropna=False
)

match_confidence
NaN     1117
High       3
Name: count, dtype: int64

In [275]:
leadershipnow_editions[
    [
        "book_id",
        "match_method",
        "match_confidence"
    ]
][
    leadershipnow_editions[
        "match_confidence"
    ].notna()
]

,book_id,match_method,match_confidence
178,BOOK00036,Exact normalized title + author,High
240,BOOK00866,Exact normalized title + author,High
261,BOOK00016,Exact normalized title + author,High


In [285]:
# =====================================================
# DEFINE MYSQL TABLE LOAD ORDER
# =====================================================

load_order = [
    "books",
    "authors",
    "categories",
    "data_sources",
    "book_authors",
    "book_categories",
    "book_editions",
    "book_metrics",
    "source_records"
]

print("MySQL load order:")

for position, table_name in enumerate(
    load_order,
    start=1
):
    print(f"{position}. {table_name}")

MySQL load order:
1. books
2. authors
3. categories
4. data_sources
5. book_authors
6. book_categories
7. book_editions
8. book_metrics
9. source_records


In [286]:
rollback_check = []

with engine.connect() as connection:

    for table_name in load_order:

        row_count = connection.execute(
            text(
                f"SELECT COUNT(*) "
                f"FROM {table_name};"
            )
        ).scalar()

        rollback_check.append({
            "table": table_name,
            "rows_after_failed_transaction": row_count
        })

rollback_check = pd.DataFrame(
    rollback_check
)

rollback_check

,table,rows_after_failed_transaction
0,books,2066
1,authors,2280
2,categories,2187
3,data_sources,2
4,book_authors,2565
5,book_categories,5926
6,book_editions,1120
7,book_metrics,950
8,source_records,2070


In [287]:
rollback_check = []

with engine.connect() as connection:

    for table_name in load_order:

        row_count = connection.execute(
            text(
                f"SELECT COUNT(*) "
                f"FROM {table_name};"
            )
        ).scalar()

        rollback_check.append({
            "table": table_name,
            "rows_after_failed_transaction": row_count
        })

rollback_check = pd.DataFrame(
    rollback_check
)

rollback_check

,table,rows_after_failed_transaction
0,books,2066
1,authors,2280
2,categories,2187
3,data_sources,2
4,book_authors,2565
5,book_categories,5926
6,book_editions,1120
7,book_metrics,950
8,source_records,2070


In [288]:
mysql_data = {
    table_name: prepare_for_mysql(dataframe)
    for table_name, dataframe in sql_tables.items()
}

In [289]:
for table_name in load_order:
    print(
        table_name,
        len(mysql_data[table_name])
    )

books 2067
authors 2281
categories 2187
data_sources 2
book_authors 2566
book_categories 5926
book_editions 1120
book_metrics 950
source_records 2070


In [290]:
rollback_check = []

with engine.connect() as connection:

    for table_name in load_order:

        row_count = connection.execute(
            text(
                f"SELECT COUNT(*) "
                f"FROM {table_name};"
            )
        ).scalar()

        rollback_check.append({
            "table": table_name,
            "mysql_rows": row_count
        })

rollback_check = pd.DataFrame(
    rollback_check
)

rollback_check

,table,mysql_rows
0,books,2066
1,authors,2280
2,categories,2187
3,data_sources,2
4,book_authors,2565
5,book_categories,5926
6,book_editions,1120
7,book_metrics,950
8,source_records,2070


In [298]:
# =====================================================
# CHECK CURRENT MYSQL TABLE COUNTS
# =====================================================

current_mysql_counts = []

with engine.connect() as connection:

    for table_name in load_order:

        row_count = connection.execute(
            text(
                f"SELECT COUNT(*) "
                f"FROM {table_name};"
            )
        ).scalar()

        current_mysql_counts.append({
            "table": table_name,
            "mysql_rows": row_count
        })

current_mysql_counts = pd.DataFrame(
    current_mysql_counts
)

display(current_mysql_counts)

,table,mysql_rows
0,books,2066
1,authors,2280
2,categories,2187
3,data_sources,2
4,book_authors,2565
5,book_categories,5926
6,book_editions,1120
7,book_metrics,950
8,source_records,2070


In [301]:
# =====================================================
# COMPARE CORRECTED PYTHON DATA WITH CURRENT MYSQL
# =====================================================

comparison = []

for table_name in load_order:

    python_rows = len(
        sql_tables[table_name]
    )

    mysql_rows = current_mysql_counts.loc[
        current_mysql_counts["table"] == table_name,
        "mysql_rows"
    ].iloc[0]

    comparison.append({
        "table": table_name,
        "python_rows_corrected": python_rows,
        "mysql_rows_current": mysql_rows,
        "difference": python_rows - mysql_rows
    })

comparison = pd.DataFrame(comparison)

display(comparison)

,table,python_rows_corrected,mysql_rows_current,difference
0,books,2067,2066,1
1,authors,2281,2280,1
2,categories,2187,2187,0
3,data_sources,2,2,0
4,book_authors,2566,2565,1
5,book_categories,5926,5926,0
6,book_editions,1120,1120,0
7,book_metrics,950,950,0
8,source_records,2070,2070,0


In [303]:
# =====================================================
# CLEAR OLD MYSQL DATA
# PRESERVE TABLE STRUCTURE AND CONSTRAINTS
# =====================================================

with engine.begin() as connection:

    connection.execute(
        text("SET FOREIGN_KEY_CHECKS = 0;")
    )

    for table_name in reversed(load_order):

        connection.execute(
            text(f"DELETE FROM {table_name};")
        )

        print(
            f"Cleared {table_name}"
        )

    connection.execute(
        text("SET FOREIGN_KEY_CHECKS = 1;")
    )

print(
    "\nOld MySQL data cleared successfully."
)

Cleared source_records
Cleared book_metrics
Cleared book_editions
Cleared book_categories
Cleared book_authors
Cleared data_sources
Cleared categories
Cleared authors
Cleared books

Old MySQL data cleared successfully.


In [304]:
# =====================================================
# VERIFY MYSQL TABLES ARE EMPTY
# =====================================================

empty_check = []

with engine.connect() as connection:

    for table_name in load_order:

        row_count = connection.execute(
            text(
                f"SELECT COUNT(*) "
                f"FROM {table_name};"
            )
        ).scalar()

        empty_check.append({
            "table": table_name,
            "rows": row_count
        })

empty_check = pd.DataFrame(
    empty_check
)

display(empty_check)

print(
    "All tables empty:",
    (empty_check["rows"] == 0).all()
)

,table,rows
0,books,0
1,authors,0
2,categories,0
3,data_sources,0
4,book_authors,0
5,book_categories,0
6,book_editions,0
7,book_metrics,0
8,source_records,0


All tables empty: True


In [305]:
# =====================================================
# PREPARE CORRECTED DATA FOR MYSQL
# =====================================================

mysql_data = {
    table_name: prepare_for_mysql(dataframe)
    for table_name, dataframe in sql_tables.items()
}

for table_name in load_order:

    print(
        f"{table_name}: "
        f"{len(mysql_data[table_name])} rows"
    )

books: 2067 rows
authors: 2281 rows
categories: 2187 rows
data_sources: 2 rows
book_authors: 2566 rows
book_categories: 5926 rows
book_editions: 1120 rows
book_metrics: 950 rows
source_records: 2070 rows


In [306]:
# =====================================================
# LOAD CORRECTED DATA INTO MYSQL
# =====================================================

with engine.begin() as connection:

    for table_name in load_order:

        dataframe = mysql_data[
            table_name
        ]

        dataframe.to_sql(
            name=table_name,
            con=connection,
            if_exists="append",
            index=False,
            chunksize=500
        )

        print(
            f"Loaded {table_name}: "
            f"{len(dataframe)} rows"
        )

print(
    "\nCorrected relational dataset loaded successfully."
)

Loaded books: 2067 rows
Loaded authors: 2281 rows
Loaded categories: 2187 rows
Loaded data_sources: 2 rows
Loaded book_authors: 2566 rows
Loaded book_categories: 5926 rows
Loaded book_editions: 1120 rows
Loaded book_metrics: 950 rows
Loaded source_records: 2070 rows

Corrected relational dataset loaded successfully.


In [307]:
# =====================================================
# FINAL PYTHON ↔ MYSQL ROW VALIDATION
# =====================================================

mysql_row_validation = []

with engine.connect() as connection:

    for table_name in load_order:

        mysql_count = connection.execute(
            text(
                f"SELECT COUNT(*) "
                f"FROM {table_name};"
            )
        ).scalar()

        python_count = len(
            sql_tables[table_name]
        )

        mysql_row_validation.append({
            "table": table_name,
            "python_rows": python_count,
            "mysql_rows": mysql_count,
            "row_match": python_count == mysql_count
        })

mysql_row_validation = pd.DataFrame(
    mysql_row_validation
)

display(mysql_row_validation)

print(
    "\nAll table counts match:",
    mysql_row_validation["row_match"].all()
)

print(
    "Total relational rows in Python:",
    mysql_row_validation["python_rows"].sum()
)

print(
    "Total relational rows in MySQL:",
    mysql_row_validation["mysql_rows"].sum()
)

,table,python_rows,mysql_rows,row_match
0,books,2067,2067,True
1,authors,2281,2281,True
2,categories,2187,2187,True
3,data_sources,2,2,True
4,book_authors,2566,2566,True
5,book_categories,5926,5926,True
6,book_editions,1120,1120,True
7,book_metrics,950,950,True
8,source_records,2070,2070,True



All table counts match: True
Total relational rows in Python: 19169
Total relational rows in MySQL: 19169


In [308]:
# =====================================================
# REFERENTIAL INTEGRITY VALIDATION
# =====================================================

integrity_queries = {
    "orphan_book_authors_books": """
        SELECT COUNT(*)
        FROM book_authors ba
        LEFT JOIN books b
            ON ba.book_id = b.book_id
        WHERE b.book_id IS NULL;
    """,

    "orphan_book_authors_authors": """
        SELECT COUNT(*)
        FROM book_authors ba
        LEFT JOIN authors a
            ON ba.author_id = a.author_id
        WHERE a.author_id IS NULL;
    """,

    "orphan_book_categories_books": """
        SELECT COUNT(*)
        FROM book_categories bc
        LEFT JOIN books b
            ON bc.book_id = b.book_id
        WHERE b.book_id IS NULL;
    """,

    "orphan_book_categories_categories": """
        SELECT COUNT(*)
        FROM book_categories bc
        LEFT JOIN categories c
            ON bc.category_id = c.category_id
        WHERE c.category_id IS NULL;
    """,

    "orphan_editions": """
        SELECT COUNT(*)
        FROM book_editions be
        LEFT JOIN books b
            ON be.book_id = b.book_id
        WHERE b.book_id IS NULL;
    """,

    "orphan_metrics": """
        SELECT COUNT(*)
        FROM book_metrics bm
        LEFT JOIN books b
            ON bm.book_id = b.book_id
        WHERE b.book_id IS NULL;
    """,

    "orphan_source_records_books": """
        SELECT COUNT(*)
        FROM source_records sr
        LEFT JOIN books b
            ON sr.book_id = b.book_id
        WHERE b.book_id IS NULL;
    """
}

integrity_results = []

with engine.connect() as connection:

    for check_name, query in integrity_queries.items():

        orphan_count = connection.execute(
            text(query)
        ).scalar()

        integrity_results.append({
            "check": check_name,
            "orphan_records": orphan_count,
            "passed": orphan_count == 0
        })

integrity_results = pd.DataFrame(
    integrity_results
)

display(integrity_results)

print(
    "\nAll referential integrity checks passed:",
    integrity_results["passed"].all()
)

,check,orphan_records,passed
0,orphan_book_authors_books,0,True
1,orphan_book_authors_authors,0,True
2,orphan_book_categories_books,0,True
3,orphan_book_categories_categories,0,True
4,orphan_editions,0,True
5,orphan_metrics,0,True
6,orphan_source_records_books,0,True



All referential integrity checks passed: True


In [309]:
# =====================================================
# BOOKS REPRESENTED BY MULTIPLE DATA SOURCES
# =====================================================

multi_source_query = """
SELECT
    sr.book_id,
    b.title,
    COUNT(DISTINCT sr.data_source_id) AS source_count
FROM source_records sr
JOIN books b
    ON sr.book_id = b.book_id
GROUP BY
    sr.book_id,
    b.title
HAVING
    COUNT(DISTINCT sr.data_source_id) > 1
ORDER BY
    sr.book_id;
"""

with engine.connect() as connection:

    multi_source_books = pd.read_sql(
        text(multi_source_query),
        connection
    )

display(multi_source_books)

print(
    "\nBooks represented by multiple sources:",
    len(multi_source_books)
)

,book_id,title,source_count
0,BOOK00016,The leadership challenge,2
1,BOOK00036,Leadership,2
2,BOOK00866,Emotional Intelligence Habits,2



Books represented by multiple sources: 3


In [311]:
# =====================================================
# INSPECT BOOK_EDITIONS SCHEMA
# =====================================================

with engine.connect() as connection:

    book_editions_schema = pd.read_sql(
        text("DESCRIBE book_editions;"),
        connection
    )

display(book_editions_schema)

,Field,Type,Null,Key,Default,Extra
0,edition_id,varchar(20),NO,PRI,None,
1,book_id,varchar(20),NO,MUL,None,
2,scrape_id,int,NO,UNI,None,
3,isbn_source,varchar(50),YES,,None,
4,isbn_normalized,varchar(20),YES,,None,
5,isbn_13,varchar(20),YES,,None,
6,isbn_status,varchar(50),YES,,None,
7,publisher,varchar(300),YES,,None,
8,publication_date,date,YES,,None,
9,publication_year,int,YES,,None,


In [314]:
# =====================================================
# VERIFY TEAM EMOTIONAL INTELLIGENCE 2.0
# USING ACTUAL MYSQL SCHEMA
# =====================================================

team_ei_query = """
SELECT
    b.book_id,
    b.title,
    GROUP_CONCAT(
        DISTINCT a.author_name
        ORDER BY a.author_name
        SEPARATOR ', '
    ) AS authors,
    be.isbn_13,
    be.publication_year,
    be.publisher,
    b.openlibrary_key,
    b.source_openlibrary,
    b.source_leadershipnow
FROM books b
LEFT JOIN book_authors ba
    ON b.book_id = ba.book_id
LEFT JOIN authors a
    ON ba.author_id = a.author_id
LEFT JOIN book_editions be
    ON b.book_id = be.book_id
WHERE
    LOWER(b.title)
    LIKE '%team emotional intelligence 2.0%'
GROUP BY
    b.book_id,
    b.title,
    be.isbn_13,
    be.publication_year,
    be.publisher,
    b.openlibrary_key,
    b.source_openlibrary,
    b.source_leadershipnow;
"""

with engine.connect() as connection:

    team_ei_mysql = pd.read_sql(
        text(team_ei_query),
        connection
    )

display(team_ei_mysql)

,book_id,title,authors,isbn_13,publication_year,publisher,openlibrary_key,source_openlibrary,source_leadershipnow
0,BOOK01189,Team Emotional Intelligence 2.0,Jean Greaves and Evan Watkins,9780974719344,2022,TalentSmart,None,0,1


In [315]:
# =====================================================
# ASSERT FALSE CROSS-SOURCE MATCH IS ABSENT
# =====================================================

assert len(team_ei_mysql) == 1

row = team_ei_mysql.iloc[0]

assert row["book_id"] == "BOOK01189"

assert str(row["isbn_13"]) == "9780974719344"

assert int(row["publication_year"]) == 2022

assert pd.isna(row["openlibrary_key"])

assert int(row["source_openlibrary"]) == 0

assert int(row["source_leadershipnow"]) == 1

print(
    "False cross-source match correction verified in MySQL."
)

False cross-source match correction verified in MySQL.


In [316]:
# =====================================================
# FINAL DATABASE AUDIT
# =====================================================

final_audit = {}

with engine.connect() as connection:

    # Total books
    final_audit["books"] = connection.execute(
        text("SELECT COUNT(*) FROM books;")
    ).scalar()

    # Total authors
    final_audit["authors"] = connection.execute(
        text("SELECT COUNT(*) FROM authors;")
    ).scalar()

    # Total editions
    final_audit["editions"] = connection.execute(
        text("SELECT COUNT(*) FROM book_editions;")
    ).scalar()

    # Total source records
    final_audit["source_records"] = connection.execute(
        text("SELECT COUNT(*) FROM source_records;")
    ).scalar()

    # Books represented by more than one source
    final_audit["multi_source_books"] = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM (
                SELECT book_id
                FROM source_records
                GROUP BY book_id
                HAVING COUNT(
                    DISTINCT data_source_id
                ) > 1
            ) AS shared_books;
        """)
    ).scalar()

final_audit_df = pd.DataFrame(
    final_audit.items(),
    columns=["metric", "value"]
)

display(final_audit_df)

# Final assertions
assert final_audit["books"] == 2067
assert final_audit["authors"] == 2281
assert final_audit["editions"] == 1120
assert final_audit["source_records"] == 2070
assert final_audit["multi_source_books"] == 3

assert mysql_row_validation[
    "row_match"
].all()

assert integrity_results[
    "passed"
].all()

print(
    "\nFINAL DATABASE VALIDATION PASSED"
)

print(
    "Python and MySQL relational data are synchronized."
)

print(
    "Cross-source entities confirmed: 3"
)

print(
    "False fuzzy match successfully excluded."
)

,metric,value
0,books,2067
1,authors,2281
2,editions,1120
3,source_records,2070
4,multi_source_books,3



FINAL DATABASE VALIDATION PASSED
Python and MySQL relational data are synchronized.
Cross-source entities confirmed: 3
False fuzzy match successfully excluded.


In [317]:
# =====================================================
# CHECK PYTHON MYSQL CONNECTION
# =====================================================

connection_query = """
SELECT
    @@hostname AS server_hostname,
    @@port AS server_port,
    USER() AS connected_user,
    CURRENT_USER() AS authenticated_user,
    @@socket AS server_socket,
    DATABASE() AS active_database;
"""

with engine.connect() as connection:
    python_mysql_connection = pd.read_sql(
        text(connection_query),
        connection
    )

display(python_mysql_connection)

print("\nSQLAlchemy connection URL:")
print(
    f"Host: {engine.url.host}"
)
print(
    f"Port: {engine.url.port}"
)
print(
    f"Database: {engine.url.database}"
)

,server_hostname,server_port,connected_user,authenticated_user,server_socket,active_database
0,Mac,3306,root@localhost,root@localhost,/tmp/mysql.sock,leadership_books_db



SQLAlchemy connection URL:
Host: localhost
Port: 3306
Database: leadership_books_db


In [318]:
server_identity_query = """
SELECT
    @@server_uuid AS server_uuid,
    @@datadir AS data_directory,
    @@version AS mysql_version,
    CONNECTION_ID() AS connection_id;
"""

with engine.connect() as connection:
    server_identity = pd.read_sql(
        text(server_identity_query),
        connection
    )

display(server_identity)

,server_uuid,data_directory,mysql_version,connection_id
0,eac9f8aa-8cf5-11f1-9369-8eadba9ad663,/opt/homebrew/var/mysql/,8.0.46,62


In [321]:
# =====================================================
# SHOW SQL TABLE COLUMNS AND DATA TYPES
# =====================================================

for table_name, df in sql_tables.items():
    print("\n" + "=" * 70)
    print(f"TABLE: {table_name}")
    print(f"Rows: {len(df):,}")
    print("=" * 70)

    print(df.dtypes)


TABLE: books
Rows: 2,067
book_id                          str
title                            str
description                      str
first_publish_year           float64
publication_year_observed    float64
cover_url                        str
openlibrary_key                  str
source_openlibrary              bool
source_leadershipnow            bool
dtype: object

TABLE: authors
Rows: 2,281
author_id               str
author_name          string
author_normalized       str
dtype: object

TABLE: book_authors
Rows: 2,566
book_id      str
author_id    str
dtype: object

TABLE: categories
Rows: 2,187
category_id               str
category_name          string
category_normalized       str
dtype: object

TABLE: book_categories
Rows: 5,926
book_id        str
category_id    str
dtype: object

TABLE: book_editions
Rows: 1,120
edition_id                      str
book_id                         str
scrape_id                     int64
isbn_source                     str
isbn_normalized    

In [322]:
for table_name, df in mysql_data.items():
    print("\n" + "=" * 70)
    print(f"TABLE: {table_name}")
    print(f"Rows: {len(df):,}")
    print("=" * 70)

    print(df.dtypes)


TABLE: books
Rows: 2,067
book_id                      object
title                        object
description                  object
first_publish_year           object
publication_year_observed    object
cover_url                    object
openlibrary_key              object
source_openlibrary           object
source_leadershipnow         object
dtype: object

TABLE: authors
Rows: 2,281
author_id            object
author_name          object
author_normalized    object
dtype: object

TABLE: book_authors
Rows: 2,566
book_id      object
author_id    object
dtype: object

TABLE: categories
Rows: 2,187
category_id            object
category_name          object
category_normalized    object
dtype: object

TABLE: book_categories
Rows: 5,926
book_id        object
category_id    object
dtype: object

TABLE: book_editions
Rows: 1,120
edition_id                  object
book_id                     object
scrape_id                   object
isbn_source                 object
isbn_normalized     

In [323]:
# =====================================================
# VERIFY DATA BEFORE MYSQL RELOAD
# =====================================================

load_order = [
    "books",
    "authors",
    "categories",
    "data_sources",
    "book_authors",
    "book_categories",
    "book_editions",
    "book_metrics",
    "source_records"
]

expected_counts = {
    "books": 2067,
    "authors": 2281,
    "categories": 2187,
    "data_sources": 2,
    "book_authors": 2566,
    "book_categories": 5926,
    "book_editions": 1120,
    "book_metrics": 950,
    "source_records": 2070
}

total_rows = 0

for table_name in load_order:
    actual = len(mysql_data[table_name])
    expected = expected_counts[table_name]

    status = "OK" if actual == expected else "CHECK"

    print(
        f"{table_name:<20} "
        f"Actual: {actual:>6,} | "
        f"Expected: {expected:>6,} | "
        f"{status}"
    )

    total_rows += actual

print("-" * 65)
print(f"TOTAL ROWS: {total_rows:,}")
print(f"EXPECTED:   {sum(expected_counts.values()):,}")

books                Actual:  2,067 | Expected:  2,067 | OK
authors              Actual:  2,281 | Expected:  2,281 | OK
categories           Actual:  2,187 | Expected:  2,187 | OK
data_sources         Actual:      2 | Expected:      2 | OK
book_authors         Actual:  2,566 | Expected:  2,566 | OK
book_categories      Actual:  5,926 | Expected:  5,926 | OK
book_editions        Actual:  1,120 | Expected:  1,120 | OK
book_metrics         Actual:    950 | Expected:    950 | OK
source_records       Actual:  2,070 | Expected:  2,070 | OK
-----------------------------------------------------------------
TOTAL ROWS: 19,169
EXPECTED:   19,169


In [325]:
# =====================================================
# LOAD CORRECTED DATA INTO MYSQL
# =====================================================

load_order = [
    "books",
    "authors",
    "categories",
    "data_sources",
    "book_authors",
    "book_categories",
    "book_editions",
    "book_metrics",
    "source_records"
]

print("Loading corrected relational data into MySQL...\n")

for table_name in load_order:

    df = mysql_data[table_name]

    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="append",
        index=False,
        method="multi",
        chunksize=500
    )

    print(f"{table_name:<20} loaded: {len(df):>6,} rows")

print("\nMySQL data load completed.")

Loading corrected relational data into MySQL...

books                loaded:  2,067 rows
authors              loaded:  2,281 rows
categories           loaded:  2,187 rows
data_sources         loaded:      2 rows
book_authors         loaded:  2,566 rows
book_categories      loaded:  5,926 rows
book_editions        loaded:  1,120 rows
book_metrics         loaded:    950 rows
source_records       loaded:  2,070 rows

MySQL data load completed.


In [326]:
# =====================================================
# FINAL MYSQL ROW COUNT VALIDATION
# =====================================================

from sqlalchemy import text

expected_counts = {
    "books": 2067,
    "authors": 2281,
    "categories": 2187,
    "data_sources": 2,
    "book_authors": 2566,
    "book_categories": 5926,
    "book_editions": 1120,
    "book_metrics": 950,
    "source_records": 2070
}

total_mysql = 0
all_valid = True

with engine.connect() as connection:
    for table_name, expected in expected_counts.items():

        actual = connection.execute(
            text(f"SELECT COUNT(*) FROM {table_name}")
        ).scalar()

        status = "OK" if actual == expected else "CHECK"

        if actual != expected:
            all_valid = False

        total_mysql += actual

        print(
            f"{table_name:<20} "
            f"MySQL: {actual:>6,} | "
            f"Expected: {expected:>6,} | "
            f"{status}"
        )

print("-" * 67)
print(f"TOTAL MYSQL ROWS: {total_mysql:,}")
print(f"EXPECTED TOTAL:   {sum(expected_counts.values()):,}")

if all_valid and total_mysql == 19169:
    print("\nFINAL ROW COUNT VALIDATION PASSED")
else:
    print("\nVALIDATION REQUIRES REVIEW")

books                MySQL:  2,067 | Expected:  2,067 | OK
authors              MySQL:  2,281 | Expected:  2,281 | OK
categories           MySQL:  2,187 | Expected:  2,187 | OK
data_sources         MySQL:      2 | Expected:      2 | OK
book_authors         MySQL:  2,566 | Expected:  2,566 | OK
book_categories      MySQL:  5,926 | Expected:  5,926 | OK
book_editions        MySQL:  1,120 | Expected:  1,120 | OK
book_metrics         MySQL:    950 | Expected:    950 | OK
source_records       MySQL:  2,070 | Expected:  2,070 | OK
-------------------------------------------------------------------
TOTAL MYSQL ROWS: 19,169
EXPECTED TOTAL:   19,169

FINAL ROW COUNT VALIDATION PASSED


In [327]:
# =====================================================
# REFERENTIAL INTEGRITY VALIDATION
# =====================================================

from sqlalchemy import text

integrity_checks = {
    "book_authors -> books": """
        SELECT COUNT(*)
        FROM book_authors ba
        LEFT JOIN books b ON ba.book_id = b.book_id
        WHERE b.book_id IS NULL
    """,

    "book_authors -> authors": """
        SELECT COUNT(*)
        FROM book_authors ba
        LEFT JOIN authors a ON ba.author_id = a.author_id
        WHERE a.author_id IS NULL
    """,

    "book_categories -> books": """
        SELECT COUNT(*)
        FROM book_categories bc
        LEFT JOIN books b ON bc.book_id = b.book_id
        WHERE b.book_id IS NULL
    """,

    "book_categories -> categories": """
        SELECT COUNT(*)
        FROM book_categories bc
        LEFT JOIN categories c ON bc.category_id = c.category_id
        WHERE c.category_id IS NULL
    """,

    "book_editions -> books": """
        SELECT COUNT(*)
        FROM book_editions be
        LEFT JOIN books b ON be.book_id = b.book_id
        WHERE b.book_id IS NULL
    """,

    "book_metrics -> books": """
        SELECT COUNT(*)
        FROM book_metrics bm
        LEFT JOIN books b ON bm.book_id = b.book_id
        WHERE b.book_id IS NULL
    """,

    "source_records -> books": """
        SELECT COUNT(*)
        FROM source_records sr
        LEFT JOIN books b ON sr.book_id = b.book_id
        WHERE b.book_id IS NULL
    """,

    "source_records -> data_sources": """
        SELECT COUNT(*)
        FROM source_records sr
        LEFT JOIN data_sources ds
            ON sr.data_source_id = ds.data_source_id
        WHERE ds.data_source_id IS NULL
    """
}

all_valid = True

with engine.connect() as connection:
    for check_name, query in integrity_checks.items():
        orphan_count = connection.execute(text(query)).scalar()

        status = "OK" if orphan_count == 0 else "CHECK"

        if orphan_count != 0:
            all_valid = False

        print(
            f"{check_name:<38} "
            f"Orphans: {orphan_count:>4} | {status}"
        )

print("-" * 60)

if all_valid:
    print("REFERENTIAL INTEGRITY VALIDATION PASSED")
else:
    print("REFERENTIAL INTEGRITY REQUIRES REVIEW")

book_authors -> books                  Orphans:    0 | OK
book_authors -> authors                Orphans:    0 | OK
book_categories -> books               Orphans:    0 | OK
book_categories -> categories          Orphans:    0 | OK
book_editions -> books                 Orphans:    0 | OK
book_metrics -> books                  Orphans:    0 | OK
source_records -> books                Orphans:    0 | OK
source_records -> data_sources         Orphans:    0 | OK
------------------------------------------------------------
REFERENTIAL INTEGRITY VALIDATION PASSED


In [328]:
# =====================================================
# FINAL CROSS-SOURCE ENTITY RESOLUTION AUDIT
# =====================================================

from sqlalchemy import text
import pandas as pd

# 1. Books represented in both sources
cross_source_query = """
SELECT
    b.book_id,
    b.title,
    b.source_openlibrary,
    b.source_leadershipnow
FROM books b
WHERE b.source_openlibrary = 1
  AND b.source_leadershipnow = 1
ORDER BY b.title;
"""

cross_source_books = pd.read_sql(
    text(cross_source_query),
    con=engine
)

print("CONFIRMED CROSS-SOURCE BOOKS")
print("=" * 70)
display(cross_source_books)

print(f"\nCross-source book count: {len(cross_source_books)}")
print("Expected: 3")


# 2. Audit Emotional Intelligence-related titles
ei_query = """
SELECT DISTINCT
    b.book_id,
    b.title,
    GROUP_CONCAT(
        DISTINCT a.author_name
        ORDER BY a.author_name
        SEPARATOR '; '
    ) AS authors,
    b.source_openlibrary,
    b.source_leadershipnow,
    be.isbn_13,
    be.publication_year
FROM books b

LEFT JOIN book_authors ba
    ON b.book_id = ba.book_id

LEFT JOIN authors a
    ON ba.author_id = a.author_id

LEFT JOIN book_editions be
    ON b.book_id = be.book_id

WHERE LOWER(b.title) LIKE '%emotional intelligence%'

GROUP BY
    b.book_id,
    b.title,
    b.source_openlibrary,
    b.source_leadershipnow,
    be.isbn_13,
    be.publication_year

ORDER BY b.title, b.book_id;
"""

ei_audit = pd.read_sql(
    text(ei_query),
    con=engine
)

print("\nEMOTIONAL INTELLIGENCE ENTITY AUDIT")
print("=" * 70)
display(ei_audit)


# 3. Explicitly verify Team Emotional Intelligence 2.0
team_ei = ei_audit[
    ei_audit["title"]
    .str.contains("Team Emotional Intelligence 2.0", case=False, na=False)
]

print("\nFALSE FUZZY MATCH CORRECTION")
print("=" * 70)

if (
    len(cross_source_books) == 3
    and len(team_ei) >= 1
    and (team_ei["source_openlibrary"] == 0).all()
    and (team_ei["source_leadershipnow"] == 1).all()
):
    print("CROSS-SOURCE ENTITY RESOLUTION VALIDATION PASSED")
    print("Confirmed cross-source entities: 3")
    print("Team Emotional Intelligence 2.0 remains LeadershipNow-only.")
    print("False fuzzy match successfully excluded.")
else:
    print("ENTITY RESOLUTION REQUIRES REVIEW")

CONFIRMED CROSS-SOURCE BOOKS


,book_id,title,source_openlibrary,source_leadershipnow
0,BOOK00866,Emotional Intelligence Habits,1,1
1,BOOK00036,Leadership,1,1
2,BOOK00016,The leadership challenge,1,1



Cross-source book count: 3
Expected: 3

EMOTIONAL INTELLIGENCE ENTITY AUDIT


,book_id,title,authors,source_openlibrary,source_leadershipnow,isbn_13,publication_year
0,BOOK00893,Applying emotional intelligence,"Joseph Ciarrochi; Mayer, John D.",1,0,NaN,NaN
1,BOOK00868,Building emotional intelligence,Linda Lantieri,1,0,NaN,NaN
2,BOOK00894,Change Your Life With Emotional Intelligence,Christine Wilding,1,0,NaN,NaN
3,BOOK00902,Communicate with emotional intelligence,John Eaton; Roy Johnson,1,0,NaN,NaN
4,BOOK00861,Emotional development and emotional intelligence,Peter Salovey,1,0,NaN,NaN
5,BOOK00856,Emotional Intelligence,Daniel Goleman,1,0,NaN,NaN
6,BOOK00858,Emotional Intelligence,Daniela GOLEMAN,1,0,NaN,NaN
7,BOOK00860,Emotional Intelligence,Christine Wilding,1,0,NaN,NaN
8,BOOK00864,Emotional intelligence,"Walton, David (Psychologist)",1,0,NaN,NaN
9,BOOK00872,Emotional intelligence,Jill Dann,1,0,NaN,NaN



FALSE FUZZY MATCH CORRECTION
CROSS-SOURCE ENTITY RESOLUTION VALIDATION PASSED
Confirmed cross-source entities: 3
Team Emotional Intelligence 2.0 remains LeadershipNow-only.
False fuzzy match successfully excluded.


In [329]:
for table_name in load_order:
    df = sql_tables[table_name]
    print(f"{table_name:<20} {len(df):>6,} rows")

books                 2,067 rows
authors               2,281 rows
categories            2,187 rows
data_sources              2 rows
book_authors          2,566 rows
book_categories       5,926 rows
book_editions         1,120 rows
book_metrics            950 rows
source_records        2,070 rows


In [330]:
from sqlalchemy import text

# Correct dependency-safe loading order
load_order = [
    "books",
    "authors",
    "categories",
    "data_sources",
    "book_authors",
    "book_categories",
    "book_editions",
    "book_metrics",
    "source_records"
]

# Safety check: MySQL tables must currently be empty
with engine.connect() as conn:
    current_counts = {
        table: conn.execute(
            text(f"SELECT COUNT(*) FROM `{table}`")
        ).scalar()
        for table in load_order
    }

print("Current MySQL row counts:")
for table, count in current_counts.items():
    print(f"{table:<20} {count:>6,}")

if any(count != 0 for count in current_counts.values()):
    raise RuntimeError(
        "STOP: At least one MySQL table is not empty. "
        "Reload cancelled to prevent duplicate records."
    )

print("\nAll destination tables are empty. Starting safe reload...\n")

# Load using APPEND only.
# Never use if_exists='replace' with the designed relational schema.
with engine.begin() as conn:
    for table in load_order:
        df = sql_tables[table]

        df.to_sql(
            name=table,
            con=conn,
            if_exists="append",
            index=False,
            method="multi",
            chunksize=500
        )

        print(f"{table:<20} loaded: {len(df):>6,} rows")

print("\nMySQL data reload completed.")

Current MySQL row counts:
books                     0
authors                   0
categories                0
data_sources              0
book_authors              0
book_categories           0
book_editions             0
book_metrics              0
source_records            0

All destination tables are empty. Starting safe reload...

books                loaded:  2,067 rows
authors              loaded:  2,281 rows
categories           loaded:  2,187 rows
data_sources         loaded:      2 rows
book_authors         loaded:  2,566 rows
book_categories      loaded:  5,926 rows
book_editions        loaded:  1,120 rows
book_metrics         loaded:    950 rows
source_records       loaded:  2,070 rows

MySQL data reload completed.


In [331]:
from sqlalchemy import text

expected_counts = {
    "books": 2067,
    "authors": 2281,
    "categories": 2187,
    "data_sources": 2,
    "book_authors": 2566,
    "book_categories": 5926,
    "book_editions": 1120,
    "book_metrics": 950,
    "source_records": 2070
}

total_mysql = 0
total_expected = sum(expected_counts.values())

with engine.connect() as conn:
    for table, expected in expected_counts.items():
        actual = conn.execute(
            text(f"SELECT COUNT(*) FROM `{table}`")
        ).scalar()

        total_mysql += actual

        status = "OK" if actual == expected else "CHECK"

        print(
            f"{table:<20} "
            f"MySQL: {actual:>6,} | "
            f"Expected: {expected:>6,} | "
            f"{status}"
        )

print("\n" + "=" * 65)
print(f"TOTAL MYSQL ROWS: {total_mysql:,}")
print(f"EXPECTED TOTAL:   {total_expected:,}")

if total_mysql == total_expected:
    print("FINAL ROW COUNT VALIDATION PASSED")
else:
    print("ROW COUNT VALIDATION FAILED")

books                MySQL:  2,067 | Expected:  2,067 | OK
authors              MySQL:  2,281 | Expected:  2,281 | OK
categories           MySQL:  2,187 | Expected:  2,187 | OK
data_sources         MySQL:      2 | Expected:      2 | OK
book_authors         MySQL:  2,566 | Expected:  2,566 | OK
book_categories      MySQL:  5,926 | Expected:  5,926 | OK
book_editions        MySQL:  1,120 | Expected:  1,120 | OK
book_metrics         MySQL:    950 | Expected:    950 | OK
source_records       MySQL:  2,070 | Expected:  2,070 | OK

TOTAL MYSQL ROWS: 19,169
EXPECTED TOTAL:   19,169
FINAL ROW COUNT VALIDATION PASSED
